# Fehlende Fußgängerüberwege aus Mapillary-Verkehrszeichen → MapRoulette

Findet Fußgängerüberweg-Zeichen (DE:350), in deren Nähe OpenStreetMap keinen Überweg
(`highway=crossing`) kennt, und macht daraus Aufgaben für die MapRoulette-Challenge 53589.
Läuft von Hand, wenn eine neue Runde fällig ist — und unverändert per nbconvert auf dem
Server, siehe [README](README.md).

Aufgebaut wie die `maproulette_tasks.ipynb` der beiden Cycleway-Kampagnen. Die Schritte
liegen in [`../campaign.py`](../campaign.py); was diese Kampagne ausmacht — Zeichen,
Abstände, Aufgabentext — in [`../pc_campaign.py`](../pc_campaign.py). Warum eine Funktion
so aussieht, steht in ihrem Docstring.

**Ablauf**

| Schritt | Funktion |
| --- | --- |
| Verkehrszeichen von data.vizsim.de spiegeln | `campaign.sync_features` |
| Überweg-Zeichen einlesen | `campaign.load_features` |
| auf Deutschland beschneiden | `campaign.clip_to_boundary` |
| frische / kurzlebige Schilder aussortieren | `campaign.filter_stable_signs` |
| Abstand zum nächsten OSM-Überweg | `campaign.distance_to_nearest` |
| Zeichen derselben Stelle zusammenfassen | `campaign.dedupe_nearby` |
| bereits bestehende Aufgaben abziehen | `campaign.drop_near_existing_tasks` |
| **neuestes** Mapillary-Bild je Zeichen holen | `campaign.newest_image_ids` |
| GeoJSON für MapRoulette schreiben | `campaign.build_maproulette_geojson` mit `pc.aufgabe` |

Geschrieben wird immer in dieselbe Datei (`maproulette_tasks_missing-ped-crossing.geojson`),
damit der Eingabepfad in MapRoulette gleich bleibt. Der letzte Abschnitt stellt den
vorherigen Stand gegenüber.

## Einstellungen

In [ ]:
import json
import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# campaign.py and pc_campaign.py live one level up. ".." works because Jupyter and
# nbconvert run the kernel in the notebook's directory - the same assumption as
# ../../output/ and ../utils/.
sys.path.insert(0, "..")
import campaign
import pc_campaign as pc

# Must match the date in ../0_prepare_osm_network.ipynb, which writes the crossings file.
set_date = "260915"

ordner_zeichen = Path("../../output")
pfad_ueberwege = Path(f"../utils/processed_osm_files/processed_crossings_germany_{set_date}.parquet")
pfad_grenze = Path("../utils/OSMB-germany.geojson.gz")
pfad_config = Path("../utils/config_mapillary_privat.json")

# Only signs last seen after this date ...
zuletzt_gesehen_nach = "2025-01-01"
# ... and seen over at least this many months - filters out temporary signs.
mindest_standzeit_monate = 12

challenge_id = 53589

# Fixed name: MapRoulette always pulls the tasks from the same path. The comparison
# at the end reads the previous round from here, too.
aufgaben_datei = Path("maproulette_tasks_missing-ped-crossing.geojson")

# A run on the server sets these from the environment instead - it must not change
# tracked files, or the deploy step stops updating the server:
#   CAMPAIGN_SYNC=0      read ../../output/ as it is. There it is what the weekly run
#                        produced, and mirroring would overwrite its manifest.
#   CAMPAIGN_OUTPUT_DIR  write the tasks there instead of next to this notebook.
sync = os.environ.get("CAMPAIGN_SYNC", "1") != "0"
ziel_geojson = Path(os.environ.get("CAMPAIGN_OUTPUT_DIR", ".")) / aufgaben_datei.name

spalten = ["id", "value", "first_seen_at", "last_seen_at", "geometry"]

print("Zeichen:", ", ".join(f"DE:{v}" for v in sorted(set(pc.UEBERWEG_ZEICHEN.values()))))
print(f"Suchradius {pc.SUCHRADIUS_M:.0f} m, zusammenfassen bis {pc.ZUSAMMENFASSEN_M:.0f} m")
print(f"spiegeln: {sync}, Aufgaben nach: {ziel_geojson}")

## 1 · Verkehrszeichen holen und einlesen

In [ ]:
if sync:
    # Fetches only what is missing locally or newer on data.vizsim.de.
    campaign.sync_features(ordner_zeichen)

manifest = json.loads((ordner_zeichen / "ml-ts_metadata.json").read_text(encoding="utf-8"))
print("Datenstand:", manifest["processed_date"])
print("im letzten Lauf unvollständig:", manifest.get("last_run_incomplete") or "nichts")

In [ ]:
# Expected number of states from the tile cache; None without one.
erwartet = campaign.count_expected_states("../../prep/tile_cache/DE-*_tiles.json")
print("Tile-Cache kennt", erwartet, "Bundesländer")

signs = campaign.load_features(
    ordner_zeichen,
    prefix=campaign.PREFIX_ZEICHEN,
    values=pc.UEBERWEG_ZEICHEN,
    columns=spalten,
    expect_files=erwartet,
)
# The parquets are cut along zoom-14 tiles, and border tiles reach abroad - where
# the OSM extract has no crossings, so every sign there would become a task.
signs = campaign.clip_to_boundary(signs, campaign.load_boundary(pfad_grenze))

## 2 · Zeitliche Filter

In [ ]:
fig, achsen = plt.subplots(1, 2, figsize=(14, 3.5))
signs["last_seen_at"].str[:7].value_counts().sort_index().plot(kind="bar", ax=achsen[0], title="alle Zeichen")

signs = campaign.filter_stable_signs(signs, zuletzt_gesehen_nach, mindest_standzeit_monate)

signs["last_seen_at"].str[:7].value_counts().sort_index().plot(
    kind="bar", ax=achsen[1], title=f"zuletzt gesehen nach {zuletzt_gesehen_nach}, ≥ {mindest_standzeit_monate} Monate"
)
plt.tight_layout()

## 3 · Abstand zum nächsten OSM-Überweg

Gemessen wird bis `pc.MESSRADIUS_M` (100 m), damit der Aufgabentext sagen kann, wie weit der
nächste gemappte Überweg entfernt ist. Zur Aufgabe wird ein Zeichen nur, wenn innerhalb von
`pc.SUCHRADIUS_M` (50 m) keiner liegt.

In [ ]:
ueberwege = gpd.read_parquet(pfad_ueberwege, columns=["geometry"])
print(f"OSM-Überwege (highway=crossing): {len(ueberwege):,}".replace(",", "."))

signs["dist_crossing_m"] = campaign.distance_to_nearest(signs, ueberwege, pc.MESSRADIUS_M)
del ueberwege

print(f"ohne OSM-Überweg in {pc.SUCHRADIUS_M:.0f} m:  {(signs.dist_crossing_m >= pc.SUCHRADIUS_M).sum():>6}")
print(f"ohne OSM-Überweg in {pc.MESSRADIUS_M:.0f} m: {(signs.dist_crossing_m == float('inf')).sum():>6}")

## 4 · Kandidaten

Ein Überweg hat meist auf jeder Straßenseite ein Zeichen. Zeichen, die höchstens
`pc.ZUSAMMENFASSEN_M` (100 m) auseinander stehen, werden zu einer Aufgabe; es bleibt das
zuletzt gesehene. Dieselbe Regel wie in der Runde 2025, die sich schneidende 50-m-Puffer
zusammenfasste.

In [ ]:
kandidaten = signs[signs["dist_crossing_m"] >= pc.SUCHRADIUS_M]
kandidaten = campaign.dedupe_nearby(kandidaten, pc.ZUSAMMENFASSEN_M).reset_index(drop=True)
kandidaten["VZ"] = kandidaten["value"].map(pc.UEBERWEG_ZEICHEN)

print(f"Kandidaten: {len(kandidaten)}")
kandidaten.plot(figsize=(6, 8), markersize=4)

## 5 · Bereits bestehende Aufgaben abziehen

Wie bei den Cycleway-Kampagnen blockiert alles außer Fixed, Created und Skipped eine
Neuanlage. Die Runde 2025 ließ jede Aufgabe blockieren, auch die erledigten: eine als Fixed
markierte Stelle, an der OSM trotzdem keinen Überweg im Umkreis von 50 m hat, kommt jetzt
wieder.

In [ ]:
# Everything but Fixed / Created / Skipped blocks a new task: what was ticked off as
# "false positive" or "already fixed" must not come back.
bestehende = campaign.load_challenge_tasks(challenge_id, campaign.load_token(pfad_config, "API_KEY_MAPROULETTE"))
print(f"blockierende Aufgaben in Challenge {challenge_id}: {len(bestehende)}")

kandidaten = campaign.drop_near_existing_tasks(kandidaten, bestehende, radius_m=pc.SUCHRADIUS_M)
print(f"neu anzulegen: {len(kandidaten)}")

## 6 · Neuestes Mapillary-Bild

Die Runde 2025 suchte das Bild über Geo-Suchen und den Blickwinkel der Kamera, mit mehreren
Anfragen je Zeichen. Die Graph-API liefert aber direkt die Bilder, in denen das Zeichen
erkannt wurde (`fields=id,images{id,captured_at}`); das neueste davon zeigt es sicher. Bis zu
50 Zeichen gehen in eine Anfrage.

In [ ]:
bilder = campaign.newest_image_ids(kandidaten["id"], campaign.load_token(pfad_config))

# Join on the string index: Mapillary ids exceed 2**53 and would be rounded as numbers.
kandidaten = kandidaten.set_index(kandidaten["id"].astype(str)).join(bilder).reset_index(drop=True)

kandidaten[["id", "last_seen_at", "dist_crossing_m", "image_id", "image_captured_at"]].head()

### Plausibilitätsprüfung

`last_seen_at` aus dem Parquet ist die letzte Aufnahme, auf der das Zeichen erkannt wurde.
Stimmt die Bildauswahl, ist das verlinkte Bild ungefähr von diesem Tag.

In [ ]:
abstand_tage = (
    kandidaten["image_captured_at"].dt.tz_localize(None) - pd.to_datetime(kandidaten["last_seen_at"])
).dt.days.abs()

print(abstand_tage.describe().to_string())
print(f"\nBild am selben Tag wie last_seen_at: {(abstand_tage <= 1).sum()} von {abstand_tage.notna().sum()}")

if abstand_tage.notna().any():
    abstand_tage.plot(kind="hist", bins=40, figsize=(8, 3), title="Tage zwischen verlinktem Bild und last_seen_at")

## 7 · GeoJSON für MapRoulette schreiben

Der bisherige Inhalt wird **vor** dem Schreiben eingelesen — die Datei behält ihren Namen,
danach gäbe es nichts mehr zu vergleichen.

In [ ]:
vorheriger_stand = campaign.read_geojson(aufgaben_datei)

aufgaben = campaign.build_maproulette_geojson(kandidaten, pc.aufgabe)
ziel_geojson.parent.mkdir(parents=True, exist_ok=True)
campaign.write_geojson(aufgaben, ziel_geojson)

if aufgaben["features"]:
    print(aufgaben["features"][0]["properties"]["instruction"])

## 8 · Vergleich mit dem vorherigen Stand

Stellt die Aufgaben-IDs der Datei im Repo denen dieses Laufs gegenüber. **anderes Bild**
zählt, wie viele Aufgaben vorher ein anderes Foto verlinkten.

In [ ]:
if vorheriger_stand:
    campaign.print_comparison(campaign.compare_task_sets(vorheriger_stand, aufgaben))
else:
    print("kein vorheriger Stand vorhanden - Vergleich übersprungen")

---

## Challenge-Beschreibung (für MapRoulette)

## 🚶 Fehlende Fußgängerüberwege anhand von Mapillary-Verkehrszeichen ergänzen (Deutschland)

Diese Challenge basiert auf automatisch erkannten **Fußgängerüberweg-Verkehrszeichen** aus
Mapillary-Bildern in Deutschland (Zeichen 350).

### 📌 Kriterien für jede Aufgabe

Nur Aufgaben, die **alle** folgenden Bedingungen erfüllen, wurden berücksichtigt:

- Das Verkehrszeichen wurde **in Mapillary erkannt**.
- Es handelt sich um ein **Fußgängerüberweg-Zeichen**
  *(Zeichen 350 – „Fußgängerüberweg“)*.
- Das Zeichen wurde über **mindestens 12 Monate hinweg** gesehen.
- Die neueste Aufnahme stammt **aus dem Jahr 2025 oder später**.
- In OpenStreetMap befindet sich **kein Fußgängerüberweg** (`highway=crossing`) **innerhalb von 50 m** des erkannten Standortes.
- Mehrere Zeichen an derselben Stelle ergeben **eine** Aufgabe.

---

### 🔍 Was du tun solltest

1. Öffne den Ort in **Mapillary** (über den Link in der Aufgabe).
2. Prüfe, ob dort tatsächlich ein **Fußgängerüberweg** existiert.
3. Falls ja, ergänze in OSM die passenden Tags, z. B.:

```
highway=crossing
crossing=uncontrolled
crossing:markings=zebra
crossing:island=no
```
*(ggf. auch `kerb=lowered`, `tactile_paving=yes`, etc.)*

4. Wenn bereits ein Fußgängerüberweg korrekt gemappt ist, kannst du die Aufgabe **als erledigt markieren**.

---

### 💡 Hinweise

- Mapillary-Bilder können veraltet oder ungenau positioniert sein – prüfe daher **immer den aktuellen Stand**.
- Fußgängerüberwege werden in OSM als **Node auf der Straßenlinie** (`highway=*`) eingetragen, nicht als eigener Weg.
- Falls kein Überweg existiert oder unklar ist, kannst du die Aufgabe **überspringen**.

---

🗺️ **Vielen Dank**, dass du hilfst, die Fußgängerinfrastruktur in OpenStreetMap zu verbessern!

---

Task-Template für MapRoulette (Feld *Instruction*):

```
{{instruction}}
```